
# 📘 Day 20: Advanced Classification (Theory + Hands-on)

This notebook is **theory-heavy** with detailed explanations and practical examples.

## Topics Covered
- Train–Test–Validation Split  
- Stratified Split  
- Cross Validation  
- Threshold Tuning  
- Multi-class (Softmax Deep Dive)  
- Multi-label Classification & Metrics  
- Hamming Loss  

---

## 🎯 Learning Goal
By the end, you will understand:
- How to correctly split data
- How to avoid bias & overfitting
- How to tune model decisions (threshold)
- Difference between multi-class vs multi-label
- How to evaluate multi-label models



# 🔹 1. Train–Test–Validation Split

## 🧠 Why is this needed?

Machine Learning models learn patterns from data.  
But if we test on the same data → model will **memorize, not generalize**.

---

## 📊 Data Splitting Strategy

| Dataset | Purpose |
|--------|--------|
| Train | Learn patterns |
| Validation | Tune hyperparameters |
| Test | Final evaluation |

---

## 💡 Key Insight

- Training data → used to **fit model**
- Validation → used to **select best model**
- Test → used only once for **final performance**

---

## ⚠️ Common Mistake

Using test data during tuning → leads to **data leakage**


In [5]:

from sklearn.model_selection import train_test_split
import numpy as np

X = np.random.rand(100, 2)
y = np.random.randint(0, 2, 100)

print('X', X[0:5])
print('y', y[0:5])

# Train + Temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)

# Validation + Test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test:", len(X_test))


X [[0.12927897 0.82462226]
 [0.01195957 0.69029075]
 [0.36910993 0.80825437]
 [0.32821884 0.44264934]
 [0.59752673 0.52118213]]
y [0 1 0 0 0]
Train: 70
Validation: 15
Test: 15



# 🔹 2. Stratified Split

## 🧠 Problem: Imbalanced Data

Example:
- Fraud = 1%
- Normal = 99%

Random split may result in:
- Test set having **0 fraud cases ❌**

---

## ✅ Solution: Stratified Split

Maintains same class distribution across splits.

---

## 💡 Why Important?

Ensures model sees **true distribution**


In [6]:

from sklearn.model_selection import train_test_split

X = np.random.rand(1000, 2)
y = np.array([0]*950 + [1]*50)  # imbalanced

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train fraud %:", sum(y_train)/len(y_train))
print("Test fraud %:", sum(y_test)/len(y_test))


Train fraud %: 0.05
Test fraud %: 0.05



# 🔹 3. Cross Validation

## 🧠 Problem

Single split may give **lucky/unlucky results**

---

## 🔄 K-Fold Cross Validation

Steps:
1. Split data into K parts
2. Train on K-1 folds
3. Test on remaining fold
4. Repeat K times

---

## 📊 Final Score = Average of all folds

---

## 💡 Benefit

Reduces variance and improves reliability


In [8]:
from sklearn.model_selection import KFold
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

kf = KFold(n_splits=5, shuffle=True, random_state=42)

model = LogisticRegression()

scores = []

for train_index, test_index in kf.split(X):
    
    # Split manually
    X_train, X_test = X[train_index], X[test_index]
    y_train, y_test = y[train_index], y[test_index]
    
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Evaluate
    acc = accuracy_score(y_test, y_pred)
    scores.append(acc)

print("Scores:", scores)
print("Mean:", np.mean(scores))

Scores: [0.95, 0.965, 0.96, 0.915, 0.96]
Mean: 0.95



# 🔹 4. Threshold Tuning

## 🧠 Default Behavior

Classifier predicts probability:
0 → 1

Default threshold = 0.5

---

## ⚖️ Why Tune?

Different business needs:

- Fraud detection → high recall
- Spam detection → high precision

---

## 💡 Tradeoff

| Threshold | Effect |
|----------|-------|
| Low | High recall |
| High | High precision |


In [9]:

import numpy as np

probs = np.array([0.2, 0.6, 0.8, 0.4])

threshold = 0.5
preds = (probs > threshold).astype(int)

print(preds)


[0 1 1 0]



# 🔹 5. Multi-class Classification (Softmax)

## 🧠 Problem

More than 2 classes

---

## 🔢 Softmax Formula

P(y=i) = e^zi / sum(e^zj)

---

## 💡 Properties

- Outputs probabilities
- Sum = 1
- Used for multi-class

---

## ⚠️ Limitation

Classes must be mutually exclusive


In [10]:

import numpy as np

logits = np.array([2.0, 1.0, 0.1])
exp = np.exp(logits)
softmax = exp / np.sum(exp)

print(softmax)


[0.65900114 0.24243297 0.09856589]



# 🔹 6. Multi-label Classification

## 🧠 Key Idea

One sample can have multiple labels.

---

## 💡 Example

Movie:
- Action
- Comedy

Both can be true

---

## ⚠️ Important

Use Sigmoid (NOT Softmax)


In [11]:
import numpy as np

# Raw scores from model (logits)
logits = np.array([2.0, 1.2, -0.5])  
# [Action, Comedy, Romance]

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

probs = sigmoid(logits)

print("Probabilities:", probs)

threshold = 0.5
predictions = (probs > threshold).astype(int)

print("Predicted Labels:", predictions)

Probabilities: [0.88079708 0.76852478 0.37754067]
Predicted Labels: [1 1 0]



# 🔹 7. Hamming Loss

## 🧠 Definition

Fraction of incorrect labels.

---

## Formula

Hamming Loss = Wrong labels / Total labels

---

## 💡 Example

True: [1,0,1]  
Pred: [1,1,0]

Wrong = 2  
Total = 3  

Loss = 2/3

---

## 🎯 Why Important?

Measures **partial correctness**


In [12]:

from sklearn.metrics import hamming_loss

y_true = [[1,0,1],[0,1,1]]
y_pred = [[1,1,1],[0,1,0]]

print("Hamming Loss:", hamming_loss(y_true, y_pred))


Hamming Loss: 0.3333333333333333
